# 03 — Track A, condition A2: label-efficiency grid

For each label budget N and seed: fine-tune BiomedCLIP on the 119 real expert captions **plus
pseudo-captions of the N labelled images**, re-encode, then train the linear heads on the **same N
images** and score them (plus zero-shot). Compared locally with A0 (frozen) and A1 (captions only).

- **Resumable:** every finished (N, seed) is saved to Drive and skipped on rerun. If Colab
  disconnects, just `Run all` again.
- **Order:** seed 0 for every N first, then seeds 1 and 2 — a partial run still gives a full curve.
- Expect roughly 2–3 hours for all 13 runs. Same one-time setup as notebooks 01/02.

In [ ]:
N_LIST = ['50', '100', '200', '500', 'all']
SEEDS = [0, 1, 2]          # 'all' uses seed 0 only (no randomness in the subset)
EPOCHS = 20

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive and copy the data to the fast local disk (fine-tuning re-reads images every epoch)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/blastocyst-clip'
LOCAL = '/content/data'
import os
if not os.path.isdir(f'{LOCAL}/blastocyst/Images'):
    !mkdir -p {LOCAL}
    !cp -r {DRIVE}/data/blastocyst {LOCAL}/
    !mkdir -p {LOCAL}/captions && cp -r {DRIVE}/data/captions/images_2 {DRIVE}/data/captions/blastocyst_captions.jsonl {LOCAL}/captions/
print(len(os.listdir(f'{LOCAL}/blastocyst/Images')), 'blastocyst images,',
      len(os.listdir(f'{LOCAL}/captions/images_2')), 'caption images on local disk')

In [ ]:
# 3. Latest code
from google.colab import userdata
REPO = '/content/blastocyst-clip'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/Daniel0813/blastocyst-clip.git'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q {url} {REPO}
del token, url
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Install
!pip install -q open_clip_torch
import torch, open_clip, sys, subprocess, json, datetime
sys.path.insert(0, REPO)
print('torch', torch.__version__, '| open_clip', open_clip.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 5. Check the caption file has the held-out split (rebuilt in notebook 02; rebuild here if not)
first = json.loads(open(f'{LOCAL}/captions/blastocyst_captions.jsonl').readline())
if 'holdout' not in first:
    !python src/select_blastocyst_captions.py --captions-dir {LOCAL}/captions
    !cp {LOCAL}/captions/blastocyst_captions.jsonl {DRIVE}/data/captions/
print('holdout field present:', 'holdout' in json.loads(open(f'{LOCAL}/captions/blastocyst_captions.jsonl').readline()))

In [ ]:
# 6. The grid (resumable). Results: {DRIVE}/results/label_efficiency/A2_*.csv
import pandas as pd
from src import zeroshot, evaluate
RES = f'{DRIVE}/results/label_efficiency'
os.makedirs(RES, exist_ok=True)
gold = evaluate.load_gold(f'{LOCAL}/blastocyst/labels.csv')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()

runs = [(n, s) for s in SEEDS for n in N_LIST if not (n == 'all' and s != SEEDS[0])]
for n, seed in runs:
    name = f'biomedclip_a2_n{n}_seed{seed}'
    done_file = f'{RES}/A2_{name}.csv'
    if os.path.exists(done_file):
        print('skip (done):', name); continue
    print(f'
===== {name}  ({datetime.datetime.now():%H:%M}) =====')
    ft_dir, emb_dir = f'{DRIVE}/results/finetune/{name}', f'/content/emb/{name}'
    n_arg = '' if n == 'all' else f'--n-labels {n}'
    !python -W ignore src/finetune.py --data-dir {LOCAL} --mode both {n_arg} --seed {seed} --epochs {EPOCHS} --out-dir {ft_dir} 2>&1 | grep -E "pairs|training|epoch  ?(1|10|20)/|saved"
    !python -W ignore src/embed.py --data-dir {LOCAL} --weights {ft_dir}/adapter.pt --out-dir {emb_dir} 2>&1 | grep -E "saved|loaded"
    # zero-shot with the adapted model (prompt matching, no heads)
    zs = zeroshot.predict(__import__('pathlib').Path(emb_dir))
    zs = zs[zs.image.isin(gold.index)]
    zs_path = f'{emb_dir}/zeroshot_pred.csv'; zs.to_csv(zs_path, header=False, index=False)
    zres = evaluate.evaluate(zs_path, gold).assign(n=n, seed=seed, condition='A2-zeroshot', emb=name)
    zres.to_csv(f'{RES}/A2zs_{name}.csv', index=False)
    # linear heads on the same N labelled images (written last = marks the run as done)
    seeds_arg = f'--seeds {seed}'
    !python -W ignore src/label_efficiency.py --condition A2 --emb-dir {emb_dir} --labels {LOCAL}/blastocyst/labels.csv --n {n} {seeds_arg} --out-dir {RES} 2>&1 | grep -E "A2 n="
    json.dump({'commit': commit, 'date': datetime.datetime.now().isoformat(timespec='seconds'),
               'gpu': torch.cuda.get_device_name(0)}, open(f'{ft_dir}/run_info.json', 'w'))
print('
all runs done' if all(os.path.exists(f"{RES}/A2_biomedclip_a2_n{n}_seed{s}.csv") for n, s in runs) else '')

When finished, download the folder `MyDrive/blastocyst-clip/results/label_efficiency` into the
local project's `results/` folder (it merges with the A0/A1 results there).